# NeuroDecode-LaBraM — Phase 5: genuine pretrained CBraMod comparator

**Development Validation only.** This first run compares CBraMod **frozen pretrained backbone** vs **full fine-tuning** on the exact frozen Phase-2 EEG HDF5 and 75/14/15 subject split. The 15-subject Test is embargoed. Do not use this notebook to score or tune on Test.

**Before starting:** Kaggle GPU enabled, Internet enabled, the previously saved 104-subject HDF5 attached via **Add Input → Notebook output**. Notebook downloads **only the 19.7 MB pretrained safetensors** (no EDF redownload). The source adapter is independently audited for amplitude scaling, 200 Hz conversion, and explicit zero padding. Experiment is one seed (42) and development-only. Source: https://github.com/mohamad679/labram-mi-project/pull/24

If this notebook fails during the pretraining key/shape audit, *do not* replace missing weights by random tensors or set `strict=False`; share the error for review.

In [ ]:
from pathlib import Path
import json, hashlib, subprocess, sys, os
ROOT = Path('/kaggle/working')
REPO = ROOT/'neurodecode_phase5_source'
RESULTS = ROOT/'NeuroDecode_Phase5_CBraMod_Validation'
ZIP = ROOT/'NeuroDecode_Phase5_CBraMod_Validation_Results.zip'
CACHE_OVERRIDE = ''  # Supply full path only if auto-discovery fails.
PINNED_REPO_COMMIT = '561870b2604d2f00198168049164337a5b9e1b16'
PINNED_RUNNER_SHA256 = 'ad14418845d5c16c0a3a9904c6edac7d5a55c4eaa9ed3f1b2ffe6c840ce3f69e'
PINNED_ADAPTER_SHA256 = '672d0cff16b758b95334e81d1b5910c23e7484297698a5d66b296838e34f0cb9'
print('Workdir:',ROOT)
print('Selected source commit:', PINNED_REPO_COMMIT)


In [ ]:
import subprocess, sys
pkgs=['braindecode[hub]==1.5.1','mne==1.13.2','scipy>=1.14,<2',
      'h5py>=3.10,<5','scikit-learn>=1.6,<2','numpy>=2,<3',
      'safetensors>=0.4,<1','huggingface_hub>=0.30,<2']
subprocess.run([sys.executable,'-m','pip','install','--quiet',*pkgs],check=True)
import torch, braindecode, scipy, numpy as np, h5py, sklearn
print({'python':sys.version.split()[0], 'torch':torch.__version__,
       'CUDA':torch.cuda.is_available(), 'braindecode':braindecode.__version__,
       'scipy':scipy.__version__, 'numpy':np.__version__})
assert torch.cuda.is_available(), 'GPU is required: set Kaggle Accelerator=GPU before this study'


In [ ]:
import subprocess, hashlib
assert not REPO.exists(), 'Source directory already exists; never silently change pinned experiment source'
subprocess.run(['git','clone','--quiet','--branch','feat/phase5-cbramod-foundation-comparator-20261013',
                '--single-branch','https://github.com/mohamad679/labram-mi-project.git',str(REPO)],check=True)
subprocess.run(['git','-C',str(REPO),'checkout','--quiet',PINNED_REPO_COMMIT],check=True)
head=subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'],text=True).strip()
assert head==PINNED_REPO_COMMIT
sha=lambda p: hashlib.sha256(Path(p).read_bytes()).hexdigest()
assert sha(REPO/'scripts/run_phase5_cbramod.py')==PINNED_RUNNER_SHA256, 'Experiment source mismatch'
assert sha(REPO/'src/phase5_cbramod.py')==PINNED_ADAPTER_SHA256, 'Input adapter source mismatch'
print('Source and adapter SHA-256 verified:',head)


In [ ]:
from pathlib import Path
candidates=[Path(CACHE_OVERRIDE)] if CACHE_OVERRIDE else (
    list(Path('/kaggle/input').rglob('physionet_mi_104subjects.h5'))+
    [Path('/kaggle/working/physionet_mi_104subjects.h5')])
candidates=[p for p in candidates if p.is_file()]
assert len(candidates)==1, ('Must locate exactly one HDF5. Found:',list(map(str,candidates)))
CACHE=candidates[0]
expected_sha='f6d2c7824f6aaa522286ea8ee5253b0220602117fc69d7b49c3af6e6a1324f7a'
assert sha(CACHE)==expected_sha, 'HDF5 dataset SHA mismatch — do not proceed'
print('Valid complete prior 104-subject HDF5:', CACHE, round(CACHE.stat().st_size/2**30,3),'GiB')


## Pre-training compatibility gate

Before any expensive full-cohort training, this runs an actual **strict pretrained safetensors** load, checks SHA and runs a shape/finite-value forward pass on a *Train-only* EEG epoch. Failure aborts rather than silently training a different architecture.

In [ ]:
import sys
sys.path.insert(0,str(REPO))
from scripts.run_phase5_cbramod import load_checkpoint_from_hf, validate_frozen_split
from src.phase5_cbramod import load_pinned_cbramod, adapt_cbramod_input
from src.portable_eeg import inspect_cache, H5SignalSource, LABELS
labels,subjects,shape=inspect_cache(CACHE, require_complete=True)
y=np.searchsorted(np.asarray(LABELS),labels)
manifest=json.loads((REPO/'configs/phase3_locked_split.json').read_text())
split=validate_frozen_split(y,subjects,manifest)
assert len(split['train'])==13050 and len(split['validation'])==2436 and len(split['test'])==2610
WEIGHTS=load_checkpoint_from_hf()
model,pretrained_info=load_pinned_cbramod(WEIGHTS,device='cpu')
source=H5SignalSource(CACHE,shape)
try:
    selected=int(split['train'][0])
    assert int(subjects[selected]) in manifest['partitions']['train']['subject_ids']
    wave=adapt_cbramod_input(source[selected])
finally:
    source.close()
with torch.inference_mode():
    logits=model(torch.from_numpy(wave[None]))
assert tuple(logits.shape)==(1,5) and bool(torch.isfinite(logits).all())
del model
print('PREFLIGHT PASS. True pretrained CBraMod loaded and train-only forward succeeded.')
print(json.dumps(pretrained_info,indent=2))


## Execute both frozen-backbone and full-backbone runs

Ten-epoch cap, 3-epoch early stopping patience, train-only class weights, head LR 0.001, backbone LR 0.00003, no Test inference. If Kaggle GPU OOM occurs, share the error; **do not silently alter hyperparameters and combine incompatible runs**.

In [ ]:
import subprocess, sys
assert not RESULTS.exists() and not ZIP.exists(), 'Results already exist; do not overwrite a completed scientific run'
cmd=[sys.executable,'-m','scripts.run_phase5_cbramod',
     '--cache',str(CACHE),'--split-manifest',str(REPO/'configs/phase3_locked_split.json'),
     '--output-dir',str(RESULTS),'--modes','probe','full','--device','cuda',
     '--seed','42','--epochs','10','--patience','3','--batch-size','8','--bootstrap','500','--amp']
logfile=ROOT/'NeuroDecode_Phase5_CBraMod_execution.log'
with logfile.open('w',encoding='utf-8') as log:
    process=subprocess.Popen(cmd,cwd=REPO,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,
                             text=True,bufsize=1)
    for line in process.stdout:
        print(line,end='')
        log.write(line)
        log.flush()
    code=process.wait()
if code:
    raise RuntimeError(f'Phase 5 runner FAILED, exit={code}; share execution log, do not report scientific PASS')
print('COMPLETE: both genuine pretrained CBraMod Validation runs finished')


In [ ]:
import zipfile, datetime
assert (RESULTS/'phase5_summary.json').is_file()
with zipfile.ZipFile(ZIP,'x',compression=zipfile.ZIP_DEFLATED,compresslevel=7) as z:
    z.write(RESULTS/'phase5_summary.json',arcname='phase5_summary.json')
    z.write(logfile,arcname='phase5_execution.log')
    for mode in ('probe','full'):
        for name in ('report.json','validation_predictions.npz'):
            f=RESULTS/mode/name
            assert f.is_file(), str(f)
            z.write(f,arcname=f'{mode}/{name}')
    z.writestr('run_provenance.json',json.dumps({
        'generated_utc':datetime.datetime.now(datetime.timezone.utc).isoformat(),
        'dataset_sha256':expected_sha,'runner_sha256':PINNED_RUNNER_SHA256,
        'adapter_sha256':PINNED_ADAPTER_SHA256,'source_commit':PINNED_REPO_COMMIT,
        'validation_only':True, 'test_set_locked':True,
    },indent=2))
with zipfile.ZipFile(ZIP) as z:
    assert z.testzip() is None
    assert all(not x.endswith(('.pt','.safetensors','.h5')) for x in z.namelist())
print('VALIDATION-ONLY ZIP READY:', ZIP)
print('ZIP bytes:', ZIP.stat().st_size,'SHA256:',sha(ZIP))


## Download and scientific handling

The ZIP deliberately excludes all EEG and large trained model weights; **save a Kaggle version of the notebook Output** if you wish to preserve exact checkpoint binaries. Send the small ZIP back for independent validation. Do not use the frozen Test split for model selection.

In [ ]:
from IPython.display import display, HTML
import base64
if ZIP.is_file():
    payload=base64.b64encode(ZIP.read_bytes()).decode('ascii')
    display(HTML(f'<a download="{ZIP.name}" href="data:application/zip;base64,{payload}" '
                 'style="display:inline-block;background:#2762a9;color:white;padding:12px;'
                 'border-radius:7px;text-decoration:none;font-weight:bold">'
                 'Download Phase 5 Results ZIP</a>'))
    print('If Safari blocks this link, refresh Kaggle Output /kaggle/working and use Download.')
else:
    print('No ZIP: previous training step failed or was not run.')
